# Day 1 - SupTech Data Ingestion and Validation

**Programme:** Operational Data Analytics and Dashboard Engineering for SupTech Systems
**Client context:** Bank of Tanzania (prototype only)

This notebook is the Day 1 reference notebook. It ingests synthetic institution
reporting records, applies a rule catalog of nineteen data quality checks, and
separates the result into four destinations: valid, flagged for review, rejected,
and exact duplicates.

> Every institution, identifier and value in this notebook is synthetic. This is not a
> Bank of Tanzania system and it contains no real institution submissions.

**What this notebook does**

1. Load a submitted CSV without repairing anything, and inspect what arrived.
2. Explain why the raw file has to stay untouched and why the original values are kept.
3. Apply a declared rule catalog and read the findings.
4. Separate records into valid, review, rejected and duplicate destinations.
5. Produce an audit trail that a supervisor can use to explain every decision.
6. Export the results in the shape a downstream API will consume.

**How to run this notebook**

- **In Colab:** run the cells from top to bottom. The setup cell finds the data for
  you if the repository is reachable; otherwise upload the three files when prompted.
- **Locally:** clone the repository, start Jupyter from the repository root, and run
  the cells from top to bottom. The data paths resolve automatically.

The reference implementation in section 4 is a readable version, written to be
read. The production version lives in `day-01-data-engineering/src/validation.py`,
and the test suite checks that the two agree on every record.

## 0. Setup

The setup cell only prepares the environment and the file paths. It does not touch
the data.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

#: Where the repository lives. Discovered automatically, or supplied by the
#: the caller when the notebook runs outside the repository.
SUPTECH_ROOT = globals().get("SUPTECH_ROOT") or os.environ.get("SUPTECH_ROOT") or None

#: Optional raw file base, for example
#: "https://raw.githubusercontent.com/your-org/your-repo/main".
REPO_RAW_BASE = os.environ.get("SUPTECH_REPO_RAW_BASE", "").strip()

DATA_FILES = {
    "reports": "day-01-data-engineering/datasets/raw/institution_reports_synthetic.csv",
    "institutions": "day-01-data-engineering/datasets/reference/institutions_registry.csv",
    "calendar": "day-01-data-engineering/datasets/reference/reporting_calendar.csv",
}

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)


def discover_root():
    """Walk up from the working directory looking for the repository."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "day-01-data-engineering").is_dir():
            return candidate
    return None


if SUPTECH_ROOT is None:
    SUPTECH_ROOT = discover_root()

print("Repository root:", SUPTECH_ROOT)

In [ ]:
# Colab only. In Colab the repository is usually not available, so upload the
# three files once. Locally this cell does nothing.
if IN_COLAB:
    from google.colab import files

    print("Upload these three files:")
    for name in DATA_FILES.values():
        print("  -", name)
    files.upload()

In [ ]:
def resolve_data_path(key):
    """Find one data file locally, then from a raw file base."""
    relative = DATA_FILES[key]

    if SUPTECH_ROOT is not None:
        candidate = Path(SUPTECH_ROOT) / relative
        if candidate.exists():
            return candidate

    if REPO_RAW_BASE:
        import urllib.request

        url = REPO_RAW_BASE.rstrip("/") + "/" + relative
        local = Path.cwd() / Path(relative).name
        urllib.request.urlretrieve(url, local)
        print("Downloaded", relative)
        return local

    raise FileNotFoundError(
        f"Could not find {relative}.\n"
        "Fix it in one of these ways:\n"
        "  1. Run this notebook from a clone of the repository.\n"
        "  2. Set SUPTECH_ROOT to the repository root.\n"
        "  3. Set SUPTECH_REPO_RAW_BASE to the raw file base of the repository.\n"
        "  4. In Colab, run the upload cell above and then re-run this cell."
    )


if "DATA_PATHS" not in globals():
    DATA_PATHS = {key: resolve_data_path(key) for key in DATA_FILES}

for key, path in DATA_PATHS.items():
    print(f"{key}: {path}")

## 1. Ingest and inspect

The first rule of a data pipeline is that the loader does not repair anything.

Notice how the file is read: every column arrives as text, and the blank value in
`total_deposits` stays blank. Nothing is converted to a number, no date is guessed,
and no value is replaced by zero. Repair belongs to the validation layer, where it
can be recorded and explained.

**Demonstration.** Load the data and inspect its shape, its types, its null
counts and its descriptive statistics.

**Exercise.** Repeat the inspection without copying the output, and write
down three observations about the file as it arrived.

In [ ]:
# The loader does not repair anything. Every column arrives as text, the blank
# value in total_deposits stays blank, and nothing is converted to a number.
# Repair belongs to the validation layer, where it can be recorded and explained.
raw = pd.read_csv(
    DATA_PATHS["reports"],
    dtype="string",
    keep_default_na=True,
    na_values=["", "NA", "N/A", "null", "NULL", "None"],
)
raw.columns = raw.columns.str.strip().str.lower()

#: Every row keeps the number of the line it came from, so that every finding
#: made later can be traced back to the source file.
raw.insert(0, "source_row_number", range(1, len(raw) + 1))

print("Loaded", len(raw), "records")
print("Columns:", list(raw.columns))

In [ ]:
print("Shape:", raw.shape)
print("Dtypes:")
print(raw.dtypes)
display(raw.head(10))

In [ ]:
print("Missing values by column:")
display(raw.isna().sum().to_frame("missing_count"))

print()
print("Exact duplicate rows:", int(raw.duplicated().sum()))
print("Distinct institutions:", raw["institution_id"].nunique())
print("Distinct reporting periods:", sorted(raw["reporting_period"].dropna().unique()))

### Design decisions to review

- An exact duplicate and a legitimate resubmission are not the same thing. What
  evidence would tell them apart?
- Why should a missing deposit amount not become a zero?
- What would a supervisor want to see alongside a rejected record?
- A `reporting_date` and a `reporting_period` both describe the same moment. Why
  keep both?

## 2. Reference data

A supervisory pipeline is never just the submissions. It also needs the reference
data that turns a number into a fact.

Two reference tables are used today:

- **`institutions_registry.csv`** - the institutions the supervisor knows about,
  their licence status, their reporting cadence, and whether they are on the
  enhanced monitoring list.
- **`reporting_calendar.csv`** - which reporting periods exist, when each one runs,
  when it is due, and whether it is open or closed.

Without the register, an unknown institution is just another row. Without the
calendar, a reporting period is only a string.

In [ ]:
INSTITUTIONS = pd.read_csv(DATA_PATHS["institutions"], dtype="string")
INSTITUTIONS.columns = INSTITUTIONS.columns.str.strip().str.lower()

CALENDAR = pd.read_csv(DATA_PATHS["calendar"], dtype="string")
CALENDAR.columns = CALENDAR.columns.str.strip().str.lower()

print("Registered institutions:", len(INSTITUTIONS))
print("Reporting periods on the calendar:", len(CALENDAR))
print()
print(INSTITUTIONS.head(5).to_string(index=False))
print()
print(CALENDAR.to_string(index=False))

## 3. The rule catalog

Validation rules are data, not code. Each rule has an identifier, a name, a
severity and a reason a supervisor would give.

Two severities:

| Severity | Meaning | Effect on the record |
| --- | --- | --- |
| `BLOCK` | The record cannot be trusted. | Removed from the curated dataset, kept with its reasons. |
| `WARN` | The record is plausible but worth a look. | Kept in the curated dataset, queued for supervisory review. |

That distinction is the point of the whole design. Not every data problem means
the same thing to a supervisor, and a pipeline that treats them the same either
discards good data or admits bad data.

In [ ]:
RULE_CATALOG = pd.DataFrame(
    [
        ("R001", "missing_required_field", "BLOCK"),
        ("R002", "invalid_institution_id_format", "BLOCK"),
        ("R003", "unknown_institution", "BLOCK"),
        ("R004", "unparseable_reporting_date", "BLOCK"),
        ("R005", "reporting_date_outside_period", "BLOCK"),
        ("R006", "invalid_reporting_period_format", "BLOCK"),
        ("R007", "reporting_period_date_mismatch", "BLOCK"),
        ("R008", "non_numeric_total_deposits", "BLOCK"),
        ("R009", "non_numeric_total_loans", "BLOCK"),
        ("R010", "negative_total_deposits", "BLOCK"),
        ("R011", "negative_total_loans", "BLOCK"),
        ("R012", "implausible_magnitude", "BLOCK"),
        ("R013", "exact_duplicate_record", "BLOCK"),
        ("R014", "duplicate_submission_key", "BLOCK"),
        ("R015", "unknown_reporting_period", "BLOCK"),
        ("W001", "loan_to_deposit_ratio_out_of_band", "WARN"),
        ("W002", "zero_total_deposits", "WARN"),
        ("W003", "institution_on_watchlist", "WARN"),
        ("W004", "reporting_frequency_granularity_mismatch", "WARN"),
    ],
    columns=["rule_id", "rule_name", "severity"],
)

print(f"{len(RULE_CATALOG)} rules: "
      f"{(RULE_CATALOG['severity'] == 'BLOCK').sum()} blocking, "
      f"{(RULE_CATALOG['severity'] == 'WARN').sum()} warnings")
print()
print(RULE_CATALOG.to_string(index=False))

## 4. Reference implementation

This is a readable reference implementation of the Day 1 pipeline, written to be read rather than
to be reused. Read it in order:

1. Load the file and keep every submitted value.
2. Separate exact duplicates before anything else runs.
3. Parse a working copy. The raw values stay available in the `raw_` columns.
4. Evaluate the rule catalog, one rule at a time.
5. Partition the records and write an audit trail.

In [ ]:
# ----------------------------------------------------------------------
# Parameters. Every threshold the pipeline uses is declared here, so the
# reviewers can argue about the number instead of hunting for it in code.
# ----------------------------------------------------------------------
INSTITUTION_ID_PATTERN = r"TZ\d{3}"
REPORTING_PERIOD_PATTERN = r"\d{4}-\d{2}"
MAX_TOTAL_DEPOSITS = 1.0e14
MAX_TOTAL_LOANS = 1.0e14
MAX_LOAN_TO_DEPOSIT_RATIO = 3.0
QUARTER_END_MONTHS = (3, 6, 9, 12)

VALUE_COLUMNS = [
    "institution_id",
    "institution_name",
    "reporting_date",
    "total_deposits",
    "total_loans",
    "reporting_period",
]
PARSED_COLUMNS = (
    "total_deposits",
    "total_loans",
    "reporting_date",
    "reporting_period",
)

BLOCK_RULES = {
    "R001", "R002", "R003", "R004", "R005", "R006", "R007", "R008",
    "R009", "R010", "R011", "R012", "R013", "R014", "R015",
}
WARN_RULES = {"W001", "W002", "W003", "W004"}

#: Rule identifiers are precise; rule names are readable. Keep both.
RULE_NAMES = dict(zip(RULE_CATALOG["rule_id"], RULE_CATALOG["rule_name"]))

# ----------------------------------------------------------------------
# 1. Keep every submitted value.
# ----------------------------------------------------------------------
working = raw.copy()
for column in VALUE_COLUMNS:
    working[f"raw_{column}"] = working[column]

audit = []

# ----------------------------------------------------------------------
# 2. Exact duplicates come out first. They need no judgement.
# ----------------------------------------------------------------------
duplicate_mask = working.duplicated(subset=VALUE_COLUMNS, keep="first")
first_row_of_group = working.groupby(
    VALUE_COLUMNS, dropna=False
)["source_row_number"].transform("min")

duplicates = working.loc[duplicate_mask].copy()
duplicates["duplicate_of_source_row_number"] = first_row_of_group[duplicate_mask]
duplicates["validation_status"] = "duplicate"

for position, row in duplicates.iterrows():
    audit.append(
        {
            "source_row_number": int(row["source_row_number"]),
            "institution_id": row["institution_id"],
            "reporting_period": row["reporting_period"],
            "rule_id": "R013",
            "severity": "BLOCK",
            "outcome": "rejected",
            "observed_value": row["institution_id"],
            "detail": f"exact duplicate of source row {int(first_row_of_group[position])}",
        }
    )

working = working.loc[~duplicate_mask].reset_index(drop=True)
reasons = [[] for _ in range(len(working))]

# ----------------------------------------------------------------------
# 3. Parse a working copy. The raw_ columns are the evidence.
# ----------------------------------------------------------------------
parsed_date = pd.to_datetime(working["reporting_date"], format="ISO8601", errors="coerce")
parsed_period = pd.to_datetime(working["reporting_period"], format="%Y-%m", errors="coerce")
numeric_deposits = pd.to_numeric(working["total_deposits"], errors="coerce")
numeric_loans = pd.to_numeric(working["total_loans"], errors="coerce")

working["reporting_date"] = parsed_date
working["reporting_period"] = parsed_period
working["total_deposits"] = numeric_deposits
working["total_loans"] = numeric_loans

# ----------------------------------------------------------------------
# 4. The audit trail and the reason list, one entry per finding.
# ----------------------------------------------------------------------
def record(rule_id, mask, detail, observed=None):
    """Register a finding for every row the mask selects."""
    hits = mask.fillna(False).astype(bool)
    for position in working.index[hits]:
        reasons[position].append(rule_id)
        audit.append(
            {
                "source_row_number": int(working.at[position, "source_row_number"]),
                "institution_id": working.at[position, "raw_institution_id"],
                "reporting_period": working.at[position, "raw_reporting_period"],
                "rule_id": rule_id,
                "severity": "BLOCK" if rule_id in BLOCK_RULES else "WARN",
                "outcome": "rejected" if rule_id in BLOCK_RULES else "review",
                "observed_value": (
                    observed.at[position] if observed is not None else None
                ),
                "detail": detail,
            }
        )


def is_blank(series):
    """True where a column is missing or contains only whitespace."""
    return series.isna() | series.fillna("").str.strip().eq("")


# ----------------------------------------------------------------------
# 5. The rules.
# ----------------------------------------------------------------------
# R001: a required field is missing, blank, or unreadable after parsing.
for column in VALUE_COLUMNS:
    blank = is_blank(working[f"raw_{column}"])
    if column in PARSED_COLUMNS:
        blank = blank | working[column].isna()
    record("R001", blank, f"{column} is missing, blank or unreadable after parsing",
           observed=working[f"raw_{column}"])

# R002 and R003: the identifier has to be well formed and known.
id_present = ~is_blank(working["raw_institution_id"])
id_format_ok = working["raw_institution_id"].str.fullmatch(
    INSTITUTION_ID_PATTERN, na=False
)
record("R002", id_present & ~id_format_ok,
       f"institution_id does not match {INSTITUTION_ID_PATTERN}",
       observed=working["raw_institution_id"])
record("R003", id_present & ~working["raw_institution_id"].isin(set(INSTITUTIONS["institution_id"])),
       "institution_id is not in the supervised institution register",
       observed=working["raw_institution_id"])

# R004, R006, R015: the date and the period have to be real.
date_present = ~is_blank(working["raw_reporting_date"])
record("R004", date_present & parsed_date.isna(),
       "reporting_date is not a valid ISO 8601 calendar date",
       observed=working["raw_reporting_date"])

period_present = ~is_blank(working["raw_reporting_period"])
# The pattern alone is not enough: "2026-13" matches the shape but is not a month.
period_format_ok = working["raw_reporting_period"].str.fullmatch(
    REPORTING_PERIOD_PATTERN, na=False
) & parsed_period.notna()
record("R006", period_present & ~period_format_ok,
       f"reporting_period does not match {REPORTING_PERIOD_PATTERN}",
       observed=working["raw_reporting_period"])

calendar_index = CALENDAR.set_index("reporting_period")
period_known = (
    period_present
    & period_format_ok
    & working["raw_reporting_period"].isin(calendar_index.index)
)
record("R015", period_present & period_format_ok & ~period_known,
       "reporting_period is not present in the reporting calendar",
       observed=working["raw_reporting_period"])

# R005: the date has to fall inside the window of the stated period.
period_start = pd.to_datetime(calendar_index["period_start"], errors="coerce")
period_end = pd.to_datetime(calendar_index["period_end"], errors="coerce")
start = working["raw_reporting_period"].map(period_start)
end = working["raw_reporting_period"].map(period_end)
record("R005",
       parsed_date.notna() & period_known & ((parsed_date < start) | (parsed_date > end)),
       "reporting_date falls outside the window of the stated reporting period",
       observed=working["raw_reporting_date"])

# R007: the period and the date have to agree about the month.
record("R007",
       parsed_date.notna() & parsed_period.notna()
       & ((parsed_date.dt.year != parsed_period.dt.year)
          | (parsed_date.dt.month != parsed_period.dt.month)),
       "reporting_period does not agree with the month of reporting_date",
       observed=working["raw_reporting_period"])

# R008 to R011: the amounts have to be numbers, and they have to be positive.
deposits_present = ~is_blank(working["raw_total_deposits"])
loans_present = ~is_blank(working["raw_total_loans"])
record("R008", deposits_present & numeric_deposits.isna(),
       "total_deposits was supplied but cannot be read as a number",
       observed=working["raw_total_deposits"])
record("R009", loans_present & numeric_loans.isna(),
       "total_loans was supplied but cannot be read as a number",
       observed=working["raw_total_loans"])
record("R010", numeric_deposits.lt(0), "total_deposits is negative",
       observed=working["raw_total_deposits"])
record("R011", numeric_loans.lt(0), "total_loans is negative",
       observed=working["raw_total_loans"])

# R012: a magnitude ceiling catches unit errors.
record("R012",
       numeric_deposits.gt(MAX_TOTAL_DEPOSITS) | numeric_loans.gt(MAX_TOTAL_LOANS),
       "a financial amount exceeds the configured plausibility ceiling",
       observed=working["raw_total_deposits"])

# R014: two different records for one institution and one period is a
# resubmission. The file alone cannot say which one is authoritative.
key_complete = working["raw_institution_id"].notna() & working["raw_reporting_period"].notna()
key_view = working.loc[key_complete]
key_duplicates = key_view.duplicated(
    subset=["raw_institution_id", "raw_reporting_period"], keep="first"
)
resubmission_mask = pd.Series(False, index=working.index)
resubmission_mask.loc[key_view.index[key_duplicates]] = True
record("R014", resubmission_mask,
       "another record was received for the same institution and reporting period",
       observed=working["raw_institution_id"])

# W001 to W004: the warnings. The record stays, a supervisor is told.
ratio = numeric_loans / numeric_deposits.where(numeric_deposits.ne(0))
record("W001",
       ratio.notna() & ratio.gt(MAX_LOAN_TO_DEPOSIT_RATIO),
       f"loan to deposit ratio is above the expected band ({MAX_LOAN_TO_DEPOSIT_RATIO})",
       observed=ratio.round(4))
record("W002", numeric_deposits.eq(0), "total_deposits is zero",
       observed=working["raw_total_deposits"])

watchlist = set(
    INSTITUTIONS.loc[
        INSTITUTIONS["watchlist_flag"].str.strip().str.lower().eq("true"),
        "institution_id",
    ]
)
record("W003", working["raw_institution_id"].isin(watchlist),
       "the reporting institution is on the enhanced monitoring list",
       observed=working["raw_institution_id"])

quarterly = set(
    INSTITUTIONS.loc[
        INSTITUTIONS["reporting_frequency"].str.strip().str.lower().eq("quarterly"),
        "institution_id",
    ]
)
record("W004",
       working["raw_institution_id"].isin(quarterly)
       & parsed_period.dt.month.notna()
       & ~parsed_period.dt.month.isin(QUARTER_END_MONTHS),
       "the reporting period is finer than the institution's quarterly cadence",
       observed=working["raw_reporting_period"])

# ----------------------------------------------------------------------
# 6. Partition.
# ----------------------------------------------------------------------
blocking = [[item for item in items if item in BLOCK_RULES] for items in reasons]
warnings = [[item for item in items if item in WARN_RULES] for items in reasons]

working["rejection_reason"] = ["; ".join(RULE_NAMES[item] for item in items) for items in blocking]
working["rejection_rule_ids"] = ["; ".join(items) for items in blocking]
working["warnings"] = ["; ".join(RULE_NAMES[item] for item in items) for items in warnings]
working["warning_rule_ids"] = ["; ".join(items) for items in warnings]

is_rejected = working["rejection_reason"].ne("")
is_review = ~is_rejected & working["warnings"].ne("")
is_valid = ~is_rejected & ~is_review

working["validation_status"] = "valid"
working.loc[is_review, "validation_status"] = "review"
working.loc[is_rejected, "validation_status"] = "rejected"

valid = working.loc[is_valid].copy()
review = working.loc[is_review].copy()
rejected = working.loc[is_rejected].copy()


def format_amount(series):
    """Present whole amounts as integers."""
    if series.dropna().mod(1).eq(0).all():
        return series.round(0).astype("Int64")
    return series.round(2)


def finish(frame, extra=()):
    """Apply the output columns and the presentation formats."""
    out = frame.copy()
    out["total_deposits"] = format_amount(out["total_deposits"])
    out["total_loans"] = format_amount(out["total_loans"])
    out["reporting_date"] = out["reporting_date"].dt.strftime("%Y-%m-%d")
    out["reporting_period"] = out["reporting_period"].dt.strftime("%Y-%m")
    columns = ["source_row_number", *VALUE_COLUMNS, "validation_status", *extra]
    return out[columns]


valid = finish(valid)
review = finish(review, extra=("warnings", "warning_rule_ids"))
rejected = finish(
    rejected,
    extra=(
        "rejection_reason",
        "rejection_rule_ids",
        *[f"raw_{column}" for column in VALUE_COLUMNS],
    ),
)
duplicates = duplicates[
    ["source_row_number", *VALUE_COLUMNS, "duplicate_of_source_row_number"]
].copy()

audit = pd.DataFrame(
    audit,
    columns=[
        "source_row_number",
        "institution_id",
        "reporting_period",
        "rule_id",
        "severity",
        "outcome",
        "observed_value",
        "detail",
    ],
)

summary = pd.DataFrame(
    {
        "metric": [
            "records_received",
            "exact_duplicates",
            "rejected_records",
            "review_records",
            "valid_records",
        ],
        "value": [
            len(raw),
            len(duplicates),
            len(rejected),
            len(review),
            len(valid),
        ],
    }
)

print("PIPELINE SUMMARY")
print(summary.to_string(index=False))

### Why the rejections look the way they do

Two patterns are worth pointing out.

**A submitted value that cannot be read produces two findings, not one.** Row 9
carries the text `not-reported` in `total_deposits`. That is a missing value
wearing a disguise, so the pipeline records both `R001` (the required field is
missing after parsing) and `R008` (the value could not be read as a number). The
audit trail keeps the original text so the supervisor can see what arrived.

**An exact duplicate and a resubmission are different problems.** Row 4 repeats row
1 exactly and is removed before anything else runs. Rows 11 and 19 repeat an
institution and a period but carry different values, so they are not duplicates.
They are resubmissions, and the file alone cannot say which version is
authoritative, so a supervisor has to confirm it.

## 5. The audit trail

The audit trail is the answer to the question a supervisor always asks: *why was
this record treated like that?*

It is written in long format, one row per finding. The same structure is loaded
into the `suptech.data_quality_audit` table on Day 2, so the dashboard can show
the reason next to the exception.

In [ ]:
print("Findings by rule:")
display(
    audit.groupby(["rule_id", "severity"])
    .size()
    .rename("records_breached")
    .reset_index()
)

print()
print("Every finding, with the value that was observed:")
display(audit.head(15))

print()
print("Rejected records and their reasons:")
display(rejected[["source_row_number", "institution_id", "rejection_reason"]])

print()
print("Records flagged for review, not rejected:")
display(review[["source_row_number", "institution_id", "warnings"]])

## 6. Export

The outputs are written in the shape the Day 2 API will consume:

| File | Contents |
| --- | --- |
| `valid_records.csv` | Records that passed every rule. |
| `review_records.csv` | Records admitted to the dataset but flagged for a supervisor. |
| `rejected_records.csv` | Records excluded, with their reasons and their original values. |
| `duplicate_records.csv` | Exact duplicates, with a pointer to the row they repeat. |
| `data_quality_audit.csv` | One row per finding. |
| `validation_summary.csv` | Counts for the run. |
| `rule_breach_counts.csv` | How many records each rule caught. |

In Colab the files are written to the runtime. Save them to Drive or download them
before the runtime is torn down, because a Colab runtime is temporary.

In [ ]:
OUTPUT_DIR = Path("day1_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

valid.to_csv(OUTPUT_DIR / "valid_records.csv", index=False)
review.to_csv(OUTPUT_DIR / "review_records.csv", index=False)
rejected.to_csv(OUTPUT_DIR / "rejected_records.csv", index=False)
duplicates.to_csv(OUTPUT_DIR / "duplicate_records.csv", index=False)
audit.to_csv(OUTPUT_DIR / "data_quality_audit.csv", index=False)
summary.to_csv(OUTPUT_DIR / "validation_summary.csv", index=False)

rule_breaches = (
    audit.groupby(["rule_id", "severity"], as_index=False)
    .size()
    .rename(columns={"size": "records_breached"})
)
rule_breaches.to_csv(OUTPUT_DIR / "rule_breach_counts.csv", index=False)

print("Written to", OUTPUT_DIR.resolve())
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {path.name}: {path.stat().st_size} bytes")

## 7. Extension exercises

Work in the cells below. Do not delete the reference implementation: it stays
visible so you can compare your rules against it.

**Task 1.** Add a rule that checks the `reporting_period` against the month of the
`reporting_date`. Name it `R016`, give it `BLOCK` severity, and add it to
`RULE_CATALOG` and to `BLOCK_RULES`. Which rows does it catch?

**Task 2.** Add a rule that rejects a `total_loans` value greater than the
`total_deposits` value by more than a factor of ten. Give it `WARN` severity,
because an unusual ratio is not proof of a bad submission.

**Task 3.** Explain, in writing, how you would distinguish a legitimate
resubmission from an exact duplicate, and what evidence you would need from the
institution to do it.

**Task 4.** Explain how you would preserve an audit trail that a supervisor could
use to defend a rejection three years later.

**Task 5.** The pipeline currently decides on a resubmission using the file alone.
What additional field in the submission would make the decision automatic, and
what would you do with the two versions in the meantime?

In [ ]:
# Task 1 and Task 2 start here.
#
# Add your rules to RULE_CATALOG and to BLOCK_RULES or WARN_RULES, then re-run
# the rule section of the reference implementation and re-check the summary.

# For example, this is how Task 1 would start:
#
#   period_month = parsed_period.dt.month
#   date_month = parsed_date.dt.month
#   record(
#       "R016",
#       parsed_period.notna() & parsed_date.notna() & (period_month != date_month),
#       "reporting_period does not match the month of reporting_date",
#       observed=working["raw_reporting_period"],
#   )
#
# Remember to add "R016" to BLOCK_RULES, otherwise the partition step will not
# treat it as a blocking finding.

print("Your rules go here. Add them to the rule section and re-run it.")

## 8. Self-check

Run this cell before committing any change. Every line has to pass.

The numbers below are the contract for the shipped dataset. If your own rules
changed the counts, the last two checks will tell you, and that is the point: a
pipeline whose output you cannot predict is not a pipeline you can defend.

In [ ]:
checks = []


def check(label, condition, detail=""):
    checks.append((label, bool(condition), detail))
    print(f"{'PASS' if condition else 'FAIL'}  {label}  {detail}")


check("every received record is accounted for",
      len(duplicates) + len(rejected) + len(review) + len(valid) == len(raw),
      f"{len(raw)} received")

check("duplicates were separated",
      len(duplicates) == 1, f"{len(duplicates)} exact duplicates")

check("rejected records carry a reason",
      rejected["rejection_reason"].ne("").all())

check("review records carry a warning",
      review["warnings"].ne("").all())

check("rejected records keep their submitted values",
      "raw_total_deposits" in rejected.columns
      and rejected.loc[rejected["source_row_number"] == 6, "raw_reporting_date"].iloc[0]
      == "invalid-date")

check("every finding is in the audit trail",
      len(audit) == int((audit["severity"] == "BLOCK").sum())
      + int((audit["severity"] == "WARN").sum()))

check("no record was lost on the way",
      len(valid) + len(review) + len(rejected) + len(duplicates) == len(raw))

failed = [label for label, passed, _ in checks if not passed]
print()
if failed:
    print("Still to fix:", failed)
else:
    print("All Day 1 checks passed.")

## Completion criteria

- [ ] Dataset loaded and inspected without repair.
- [ ] Reference data loaded and explained.
- [ ] Rule catalog read and discussed.
- [ ] Numeric fields and reporting dates validated.
- [ ] Valid, review, rejected and duplicate records separated.
- [ ] Rejection reasons and warnings generated.
- [ ] Audit trail produced.
- [ ] Outputs exported.
- [ ] Extension exercises attempted.
- [ ] Self-check passed.

**Carry into Day 2:** `valid_records.csv` and `review_records.csv` are the inputs
for the supervisory indicators, and `data_quality_audit.csv` is what makes an
exception on the dashboard explainable.